In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive'))

Mounted at /content/drive
['有关我国疫情防控的法律思考（大作业）.docx', 'ZLibrary', 'MATLAB教程 (张志涌 杨祖樱) (z-lib.org).pdf', 'Colab Notebooks', '.ipynb_checkpoints', 'PyTorch_CNN', 'dl', 'gcn', 'Assignment 3 - Bundle Adjustment', '3dgs_final_result.zip', 'my_model.ply', 'my_model1.ply', 'my_model2.ply', 'my_model3.ply', 'my_model4.ply', 'my_model5.ply', 'Assignment 4 - 3DGS']


In [12]:
!apt-get update -y
!apt-get install -y colmap

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:13 http://archive.ubuntu.com/ubuntu jam

In [ ]:
import os
import subprocess

os.environ.setdefault('QT_QPA_PLATFORM', 'offscreen')

data_dir = '/content/drive/MyDrive/Assignment 4 - 3DGS/data/chair'
db_path = os.path.join(data_dir, 'database.db')

subprocess.run(['colmap', 'feature_extractor',
                '--image_path', os.path.join(data_dir, 'images'),
                '--database_path', db_path,
                '--ImageReader.single_camera', '1',
                '--ImageReader.camera_model', 'PINHOLE',
                '--SiftExtraction.use_gpu', '0'], check=True)
print("提取完成！")

subprocess.run(['colmap', 'sequential_matcher',
                '--database_path', db_path,
                '--SiftMatching.use_gpu', '0'], check=True)
print("匹配完成！")

os.makedirs(os.path.join(data_dir, 'sparse'), exist_ok=True)
subprocess.run(['colmap', 'mapper',
                '--image_path', os.path.join(data_dir, 'images'),
                '--database_path', db_path,
                '--output_path', os.path.join(data_dir, 'sparse')], check=True)
print("重建完成！")

os.makedirs(os.path.join(data_dir, 'sparse', '0_text'), exist_ok=True)
subprocess.run(['colmap', 'model_converter',
                '--input_path', os.path.join(data_dir, 'sparse', '0'),
                '--output_path', os.path.join(data_dir, 'sparse', '0_text'),
                '--output_type', 'TXT'], check=True)
print("准备进入 3DGS 训练阶段")

⏳ [1/4] 提取特征 (纯 CPU) ...
✅ 提取完成！
⏳ [2/4] 顺序匹配 (纯 CPU，极速版) ...
✅ 匹配完成！
⏳ [3/4] 稀疏重建 ...
✅ 重建完成！
⏳ [4/4] 格式转换 ...
✅ 全部搞定！准备进入 3DGS 训练阶段！


In [ ]:
import numpy as np
import cv2
import os

def qvec2rotmat(qvec):
    """Convert quaternion to rotation matrix"""
    return np.array([
        [1 - 2 * qvec[2]**2 - 2 * qvec[3]**2,
         2 * qvec[1] * qvec[2] - 2 * qvec[0] * qvec[3],
         2 * qvec[3] * qvec[1] + 2 * qvec[0] * qvec[2]],
        [2 * qvec[1] * qvec[2] + 2 * qvec[0] * qvec[3],
         1 - 2 * qvec[1]**2 - 2 * qvec[3]**2,
         2 * qvec[2] * qvec[3] - 2 * qvec[0] * qvec[1]],
        [2 * qvec[3] * qvec[1] - 2 * qvec[0] * qvec[2],
         2 * qvec[2] * qvec[3] + 2 * qvec[0] * qvec[1],
         1 - 2 * qvec[1]**2 - 2 * qvec[2]**2]])

def read_points3D_text(path):
    """Read points3D.txt file"""
    points3D = {}
    with open(path, 'r') as f:
        for line in f:
            if line[0] == '#':
                continue
            data = line.split()
            point_id = int(data[0])
            xyz = np.array([float(x) for x in data[1:4]])
            rgb = np.array([int(x) for x in data[4:7]])
            error = float(data[7])
            points3D[point_id] = {
                'xyz': xyz,
                'rgb': rgb,
                'error': error
            }
    return points3D

def read_images_text(path):
    """Read images.txt file"""
    images = {}
    with open(path, 'r') as f:
        lines = f.readlines()

    for i in range(0, len(lines), 2):
        line = lines[i]
        if line[0] == '#':
            continue
        data = line.split()
        image_id = int(data[0])
        qvec = np.array([float(x) for x in data[1:5]])
        tvec = np.array([float(x) for x in data[5:8]])
        camera_id = int(data[8])
        name = data[9]

        R = qvec2rotmat(qvec)

        images[image_id] = {
            'R': R,
            't': tvec.reshape(3,1),
            'camera_id': camera_id,
            'name': name
        }
    return images

def read_cameras_text(path):
    """Read cameras.txt file"""
    cameras = {}
    with open(path, 'r') as f:
        for line in f:
            if line[0] == '#':
                continue
            data = line.split()
            camera_id = int(data[0])
            model = data[1]
            width = int(data[2])
            height = int(data[3])
            params = np.array([float(x) for x in data[4:]])
            cameras[camera_id] = {
                'model': model,
                'width': width,
                'height': height,
                'params': params
            }
    return cameras

def get_intrinsic_matrix(camera):
    """Get intrinsic matrix from camera parameters"""
    if camera['model'] == 'PINHOLE':
        fx, fy, cx, cy = camera['params']
        K = np.array([[fx, 0, cx],
                     [0, fy, cy],
                     [0, 0, 1]])
        return K
    else:
        raise ValueError(f"Camera model {camera['model']} not supported yet")

def project_points(points3D, R, t, K):
    """Project 3D points to image plane"""
    # Convert points to camera coordinates
    points3D_cam = (R @ points3D.T + t).T

    # Get points in front of camera
    mask = points3D_cam[:, 2] > 0

    # Project to image plane
    points3D_cam = points3D_cam[mask]
    points2D = points3D_cam[:, :2] / points3D_cam[:, 2:]
    points2D = (K[:2, :2] @ points2D.T).T + K[:2, 2]

    return points2D, mask

def main():
    dataset_path = '/content/drive/MyDrive/Assignment 4 - 3DGS/data/chair'

    sparse_path = os.path.join(dataset_path, "sparse", "0_text")
    images_dir = os.path.join(dataset_path, "images")
    output_dir = os.path.join(dataset_path, "projections")  # Directory for output images

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Load COLMAP data
    print("Loading COLMAP data...")
    cameras = read_cameras_text(os.path.join(sparse_path, "cameras.txt"))
    images = read_images_text(os.path.join(sparse_path, "images.txt"))
    points3D = read_points3D_text(os.path.join(sparse_path, "points3D.txt"))

    # Convert points3D to numpy arrays for efficient processing
    points3D_xyz = np.array([p['xyz'] for p in points3D.values()])
    points3D_rgb = np.array([p['rgb'] for p in points3D.values()])

    # Process each image
    print("Processing images...")
    for image_id, image_data in images.items():
        # Get image path
        image_name = image_data['name']
        image_path = os.path.join(images_dir, image_name)

        # Skip if image doesn't exist
        if not os.path.exists(image_path):
            print(f"Warning: Image {image_name} not found")
            continue

        # Load image
        img = cv2.imread(image_path)
        if img is None:
            print(f"Warning: Could not load image {image_name}")
            continue
        img_ori = np.array(img)
        img[:] = 0

        # Get camera parameters
        camera = cameras[image_data['camera_id']]
        K = get_intrinsic_matrix(camera)
        R = image_data['R']
        t = image_data['t']

        # Project points
        points2D, mask = project_points(points3D_xyz, R, t, K)

        # Draw points on image
        points2D = points2D.astype(int)
        colors = points3D_rgb[mask]

        for pt, color in zip(points2D, colors):
            # Check if point is within image bounds
            if 0 <= pt[0] < img.shape[1] and 0 <= pt[1] < img.shape[0]:
                cv2.circle(img, (pt[0], pt[1]), 2, color[::-1].tolist(), -1)  # BGR to RGB

        # Save result
        output_path = os.path.join(output_dir, image_name)
        com_img = np.concatenate((img_ori, img), axis=1)
        com_img = cv2.resize(com_img, (0,0), fx=0.125, fy=0.125)
        cv2.imwrite(output_path, com_img)
        print(f"Processed {image_name}")

    print("\n Done! Check the 'projections' folder in your Google Drive for results.")

if __name__ == "__main__":
    main()

Loading COLMAP data...
Processing images...
Processed r_99.png
Processed r_97.png
Processed r_98.png
Processed r_94.png
Processed r_96.png
Processed r_95.png
Processed r_92.png
Processed r_93.png
Processed r_90.png
Processed r_91.png
Processed r_88.png
Processed r_9.png
Processed r_89.png
Processed r_87.png
Processed r_85.png
Processed r_86.png
Processed r_83.png
Processed r_82.png
Processed r_84.png
Processed r_8.png
Processed r_80.png
Processed r_78.png
Processed r_77.png
Processed r_81.png
Processed r_79.png
Processed r_75.png
Processed r_76.png
Processed r_74.png
Processed r_72.png
Processed r_73.png
Processed r_70.png
Processed r_71.png
Processed r_67.png
Processed r_7.png
Processed r_66.png
Processed r_69.png
Processed r_68.png
Processed r_65.png
Processed r_63.png
Processed r_64.png
Processed r_62.png
Processed r_35.png
Processed r_31.png
Processed r_32.png
Processed r_33.png
Processed r_30.png
Processed r_26.png
Processed r_3.png
Processed r_28.png
Processed r_22.png
Processed 

In [ ]:
!apt-get install python3.10 python3.10-distutils -y -qq
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

!python3.10 -m pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118
!python3.10 -m pip install -q fvcore iopath portalocker opencv-python plyfile tqdm

!python3.10 -m pip install -q https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu118_pyt210/pytorch3d-0.7.5-cp310-cp310-linux_x86_64.whl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wheel]
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
!python3.10 -m pip install -q "opencv-python<4.10" "numpy<2" natsort

In [ ]:
%cd "/content/drive/MyDrive/Assignment 4 - 3DGS"

!python3.10 train.py --colmap_dir data/chair --checkpoint_dir data/chair/checkpoints

/content/drive/MyDrive/Assignment 4 - 3DGS
Using device: cuda
init_scales tensor(0.0576) tensor(0.3165)
Starting training...
Training on 100 images for 200 epochs
Debug images will be saved every 1 epochs
Using 4 debug samples
Epoch 0: 100% 100/100 [00:39<00:00,  2.50it/s, loss=0.0793]
Epoch 1: 100% 100/100 [00:08<00:00, 12.28it/s, loss=0.0572]
Epoch 2: 100% 100/100 [00:08<00:00, 12.22it/s, loss=0.0474]
Epoch 3: 100% 100/100 [00:08<00:00, 12.28it/s, loss=0.0428]
Epoch 4: 100% 100/100 [00:08<00:00, 12.27it/s, loss=0.0402]
Epoch 5: 100% 100/100 [00:08<00:00, 12.25it/s, loss=0.0385]
Epoch 6: 100% 100/100 [00:08<00:00, 12.28it/s, loss=0.0372]
Epoch 7: 100% 100/100 [00:08<00:00, 12.24it/s, loss=0.0362]
Epoch 8: 100% 100/100 [00:08<00:00, 12.27it/s, loss=0.0355]
Epoch 9: 100% 100/100 [00:08<00:00, 12.31it/s, loss=0.0350]
Epoch 10: 100% 100/100 [00:08<00:00, 12.28it/s, loss=0.0345]
Epoch 11: 100% 100/100 [00:08<00:00, 12.30it/s, loss=0.0340]
Epoch 12: 100% 100/100 [00:08<00:00, 12.28it/s, los

In [10]:
%cd "/content/drive/MyDrive/Assignment 4 - 3DGS"

!python3.10 render_3dgs_mv.py \
    --colmap_dir data/chair \
    --checkpoint data/chair/checkpoints/checkpoint_000180.pt \
    --num_frames 240 \
    --fps 30

/content/drive/MyDrive/Assignment 4 - 3DGS
Using device: cuda
init_scales tensor(0.0576) tensor(0.3165)
Building horizontal orbit from 100 training cameras → 240 frames
Rendering: 100% 240/240 [00:06<00:00, 39.22it/s]
Video saved to: data/chair/render_mv.mp4


In [ ]:
import os
import subprocess

os.environ.setdefault('QT_QPA_PLATFORM', 'offscreen')

data_dir = '/content/drive/MyDrive/Assignment 4 - 3DGS/data/lego'
db_path = os.path.join(data_dir, 'database.db')

subprocess.run(['colmap', 'feature_extractor',
                '--image_path', os.path.join(data_dir, 'images'),
                '--database_path', db_path,
                '--ImageReader.single_camera', '1',
                '--ImageReader.camera_model', 'PINHOLE',
                '--SiftExtraction.use_gpu', '0'], check=True)
print("提取完成！")

subprocess.run(['colmap', 'exhaustive_matcher',
                '--database_path', db_path,
                '--SiftMatching.use_gpu', '0'], check=True)
print("匹配完成！")

os.makedirs(os.path.join(data_dir, 'sparse'), exist_ok=True)
subprocess.run(['colmap', 'mapper',
                '--image_path', os.path.join(data_dir, 'images'),
                '--database_path', db_path,
                '--output_path', os.path.join(data_dir, 'sparse')], check=True)
print("重建完成！")

os.makedirs(os.path.join(data_dir, 'sparse', '0_text'), exist_ok=True)
subprocess.run(['colmap', 'model_converter',
                '--input_path', os.path.join(data_dir, 'sparse', '0'),
                '--output_path', os.path.join(data_dir, 'sparse', '0_text'),
                '--output_type', 'TXT'], check=True)
print("准备进入 3DGS 训练阶段")

⏳ [1/4] 提取特征 (纯 CPU) ...
✅ 提取完成！
⏳ [2/4] 顺序匹配 (纯 CPU，极速版) ...
✅ 匹配完成！
⏳ [3/4] 稀疏重建 ...
✅ 重建完成！
⏳ [4/4] 格式转换 ...
✅ 全部搞定！准备进入 3DGS 训练阶段！


In [ ]:
import numpy as np
import cv2
import os

def qvec2rotmat(qvec):
    """Convert quaternion to rotation matrix"""
    return np.array([
        [1 - 2 * qvec[2]**2 - 2 * qvec[3]**2,
         2 * qvec[1] * qvec[2] - 2 * qvec[0] * qvec[3],
         2 * qvec[3] * qvec[1] + 2 * qvec[0] * qvec[2]],
        [2 * qvec[1] * qvec[2] + 2 * qvec[0] * qvec[3],
         1 - 2 * qvec[1]**2 - 2 * qvec[3]**2,
         2 * qvec[2] * qvec[3] - 2 * qvec[0] * qvec[1]],
        [2 * qvec[3] * qvec[1] - 2 * qvec[0] * qvec[2],
         2 * qvec[2] * qvec[3] + 2 * qvec[0] * qvec[1],
         1 - 2 * qvec[1]**2 - 2 * qvec[2]**2]])

def read_points3D_text(path):
    """Read points3D.txt file"""
    points3D = {}
    with open(path, 'r') as f:
        for line in f:
            if line[0] == '#':
                continue
            data = line.split()
            point_id = int(data[0])
            xyz = np.array([float(x) for x in data[1:4]])
            rgb = np.array([int(x) for x in data[4:7]])
            error = float(data[7])
            points3D[point_id] = {
                'xyz': xyz,
                'rgb': rgb,
                'error': error
            }
    return points3D

def read_images_text(path):
    """Read images.txt file"""
    images = {}
    with open(path, 'r') as f:
        lines = f.readlines()

    for i in range(0, len(lines), 2):
        line = lines[i]
        if line[0] == '#':
            continue
        data = line.split()
        image_id = int(data[0])
        qvec = np.array([float(x) for x in data[1:5]])
        tvec = np.array([float(x) for x in data[5:8]])
        camera_id = int(data[8])
        name = data[9]

        R = qvec2rotmat(qvec)

        images[image_id] = {
            'R': R,
            't': tvec.reshape(3,1),
            'camera_id': camera_id,
            'name': name
        }
    return images

def read_cameras_text(path):
    """Read cameras.txt file"""
    cameras = {}
    with open(path, 'r') as f:
        for line in f:
            if line[0] == '#':
                continue
            data = line.split()
            camera_id = int(data[0])
            model = data[1]
            width = int(data[2])
            height = int(data[3])
            params = np.array([float(x) for x in data[4:]])
            cameras[camera_id] = {
                'model': model,
                'width': width,
                'height': height,
                'params': params
            }
    return cameras

def get_intrinsic_matrix(camera):
    """Get intrinsic matrix from camera parameters"""
    if camera['model'] == 'PINHOLE':
        fx, fy, cx, cy = camera['params']
        K = np.array([[fx, 0, cx],
                     [0, fy, cy],
                     [0, 0, 1]])
        return K
    else:
        raise ValueError(f"Camera model {camera['model']} not supported yet")

def project_points(points3D, R, t, K):
    """Project 3D points to image plane"""
    # Convert points to camera coordinates
    points3D_cam = (R @ points3D.T + t).T

    # Get points in front of camera
    mask = points3D_cam[:, 2] > 0

    # Project to image plane
    points3D_cam = points3D_cam[mask]
    points2D = points3D_cam[:, :2] / points3D_cam[:, 2:]
    points2D = (K[:2, :2] @ points2D.T).T + K[:2, 2]

    return points2D, mask

def main():
    dataset_path = '/content/drive/MyDrive/Assignment 4 - 3DGS/data/lego'

    sparse_path = os.path.join(dataset_path, "sparse", "0_text")
    images_dir = os.path.join(dataset_path, "images")
    output_dir = os.path.join(dataset_path, "projections")  # Directory for output images

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Load COLMAP data
    print("Loading COLMAP data...")
    cameras = read_cameras_text(os.path.join(sparse_path, "cameras.txt"))
    images = read_images_text(os.path.join(sparse_path, "images.txt"))
    points3D = read_points3D_text(os.path.join(sparse_path, "points3D.txt"))

    # Convert points3D to numpy arrays for efficient processing
    points3D_xyz = np.array([p['xyz'] for p in points3D.values()])
    points3D_rgb = np.array([p['rgb'] for p in points3D.values()])

    # Process each image
    print("Processing images...")
    for image_id, image_data in images.items():
        # Get image path
        image_name = image_data['name']
        image_path = os.path.join(images_dir, image_name)

        # Skip if image doesn't exist
        if not os.path.exists(image_path):
            print(f"Warning: Image {image_name} not found")
            continue

        # Load image
        img = cv2.imread(image_path)
        if img is None:
            print(f"Warning: Could not load image {image_name}")
            continue
        img_ori = np.array(img)
        img[:] = 0

        # Get camera parameters
        camera = cameras[image_data['camera_id']]
        K = get_intrinsic_matrix(camera)
        R = image_data['R']
        t = image_data['t']

        # Project points
        points2D, mask = project_points(points3D_xyz, R, t, K)

        # Draw points on image
        points2D = points2D.astype(int)
        colors = points3D_rgb[mask]

        for pt, color in zip(points2D, colors):
            # Check if point is within image bounds
            if 0 <= pt[0] < img.shape[1] and 0 <= pt[1] < img.shape[0]:
                cv2.circle(img, (pt[0], pt[1]), 2, color[::-1].tolist(), -1)  # BGR to RGB

        # Save result
        output_path = os.path.join(output_dir, image_name)
        com_img = np.concatenate((img_ori, img), axis=1)
        com_img = cv2.resize(com_img, (0,0), fx=0.125, fy=0.125)
        cv2.imwrite(output_path, com_img)
        print(f"Processed {image_name}")

    print("\n Done! Check the 'projections' folder in your Google Drive for results.")

if __name__ == "__main__":
    main()

Loading COLMAP data...
Processing images...
Processed r_99.png
Processed r_96.png
Processed r_97.png
Processed r_98.png
Processed r_94.png
Processed r_93.png
Processed r_95.png
Processed r_90.png
Processed r_91.png
Processed r_92.png
Processed r_9.png
Processed r_88.png
Processed r_86.png
Processed r_87.png
Processed r_89.png
Processed r_83.png
Processed r_84.png
Processed r_80.png
Processed r_82.png
Processed r_85.png
Processed r_81.png
Processed r_8.png
Processed r_79.png
Processed r_78.png
Processed r_77.png
Processed r_73.png
Processed r_76.png
Processed r_74.png
Processed r_71.png
Processed r_7.png
Processed r_70.png
Processed r_72.png
Processed r_75.png
Processed r_69.png
Processed r_68.png
Processed r_67.png
Processed r_66.png
Processed r_63.png
Processed r_62.png
Processed r_58.png
Processed r_61.png
Processed r_35.png
Processed r_33.png
Processed r_30.png
Processed r_29.png
Processed r_3.png
Processed r_31.png
Processed r_32.png
Processed r_28.png
Processed r_27.png
Processed 

In [18]:
%cd "/content/drive/MyDrive/Assignment 4 - 3DGS"

!python3.10 train.py --colmap_dir data/lego --checkpoint_dir data/lego/checkpoints --num_epochs 100

/content/drive/MyDrive/Assignment 4 - 3DGS
Using device: cuda
init_scales tensor(0.1022) tensor(0.5872)
Starting training...
Training on 100 images for 100 epochs
Debug images will be saved every 1 epochs
Using 4 debug samples
Epoch 0: 100% 100/100 [00:02<00:00, 41.21it/s, loss=0.1173]
Epoch 1: 100% 100/100 [00:02<00:00, 48.29it/s, loss=0.0794]
Epoch 2: 100% 100/100 [00:02<00:00, 47.94it/s, loss=0.0614]
Epoch 3: 100% 100/100 [00:02<00:00, 48.07it/s, loss=0.0546]
Epoch 4: 100% 100/100 [00:02<00:00, 46.89it/s, loss=0.0511]
Epoch 5: 100% 100/100 [00:02<00:00, 46.96it/s, loss=0.0490]
Epoch 6: 100% 100/100 [00:02<00:00, 45.88it/s, loss=0.0475]
Epoch 7: 100% 100/100 [00:02<00:00, 48.05it/s, loss=0.0464]
Epoch 8: 100% 100/100 [00:02<00:00, 46.90it/s, loss=0.0454]
Epoch 9: 100% 100/100 [00:02<00:00, 48.38it/s, loss=0.0447]
Epoch 10: 100% 100/100 [00:02<00:00, 47.68it/s, loss=0.0440]
Epoch 11: 100% 100/100 [00:02<00:00, 46.40it/s, loss=0.0435]
Epoch 12: 100% 100/100 [00:02<00:00, 45.88it/s, los

In [20]:
%cd "/content/drive/MyDrive/Assignment 4 - 3DGS"

!python3.10 render_3dgs_mv.py \
    --colmap_dir data/lego \
    --checkpoint data/lego/checkpoints/checkpoint_000080.pt \
    --num_frames 240 \
    --fps 30

/content/drive/MyDrive/Assignment 4 - 3DGS
Using device: cuda
init_scales tensor(0.1022) tensor(0.5872)
Building horizontal orbit from 100 training cameras → 240 frames
Rendering: 100% 240/240 [00:01<00:00, 140.20it/s]
Video saved to: data/lego/render_mv.mp4


In [21]:
import os
import subprocess

data_dir = '/content/drive/MyDrive/Assignment 4 - 3DGS/data/playroom'

os.makedirs(os.path.join(data_dir, 'sparse', '0_text'), exist_ok=True)

subprocess.run(['colmap', 'model_converter',
                '--input_path', os.path.join(data_dir, 'sparse', '0'),
                '--output_path', os.path.join(data_dir, 'sparse', '0_text'),
                '--output_type', 'TXT'], check=True)

print("Please check the folder playroom/sparse/0_text")

Please check the folder playroom/sparse/0_text


In [24]:
%cd "/content/drive/MyDrive/Assignment 4 - 3DGS"

!python3.10 train.py --colmap_dir data/playroom --checkpoint_dir data/playroom/checkpoints --num_epochs 50

/content/drive/MyDrive/Assignment 4 - 3DGS
Using device: cuda
init_scales tensor(0.0942) tensor(0.9200)
Starting training...
Training on 225 images for 50 epochs
Debug images will be saved every 1 epochs
Using 4 debug samples
Epoch 0: 100% 225/225 [01:29<00:00,  2.53it/s, loss=0.4335]
Epoch 1: 100% 225/225 [01:28<00:00,  2.53it/s, loss=nan]
/content/drive/MyDrive/Assignment 4 - 3DGS/train.py:70: RuntimeWarning: invalid value encountered in cast
  r = (rendered[b] * 255).clip(0, 255).astype(np.uint8)
Epoch 2: 100% 225/225 [01:28<00:00,  2.53it/s, loss=nan]
Epoch 3: 100% 225/225 [01:28<00:00,  2.53it/s, loss=nan]
Epoch 4:  12% 27/225 [00:10<01:18,  2.54it/s, loss=nan]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e7acee82950>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1478, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/data

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install python3.10 python3.10-dev python3.10-distutils -y -qq
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

!python3.10 -m pip install -q numpy==1.26.4 plyfile tqdm
!python3.10 -m pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118

!python3.10 -m pip install -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl
!python3.10 -m pip install -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/simple_knn-0.0.0-cp310-cp310-linux_x86_64.whl

%cd /content
!rm -rf gaussian-splatting
!git clone https://github.com/camenduru/gaussian-splatting

%cd /content/gaussian-splatting

!python3.10 train.py \
  -s "/content/drive/MyDrive/Assignment 4 - 3DGS/data/lego" \
  -m "/content/drive/MyDrive/Assignment 4 - 3DGS/result/lego_official" \
  --iterations 30000

/content
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3.10-dev.
(Reading database ... 123756 files and directories currently installed.)
Preparing to unpack .../python3.10-dev_3.10.12-1~22.04.15_amd64.deb ...
Unpacking python3.10-dev (3.10.12-1~22.04.15) ...
Setting up python3.10-dev (3.10.12-1~22.04.15) ...
Processing triggers for man-db (2.10.2-1) ...
  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1
/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 

In [26]:
%cd /content/gaussian-splatting

!python3.10 train.py \
  -s "/content/drive/MyDrive/Assignment 4 - 3DGS/data/chair" \
  -m "/content/drive/MyDrive/Assignment 4 - 3DGS/result/chair_official" \
  --iterations 30000

/content/gaussian-splatting
Optimizing /content/drive/MyDrive/Assignment 4 - 3DGS/result/chair_official
Output folder: /content/drive/MyDrive/Assignment 4 - 3DGS/result/chair_official [17/05 15:14:07]
Tensorboard not available: not logging progress [17/05 15:14:07]
Reading camera 100/100 [17/05 15:14:10]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [17/05 15:14:10]
Loading Training Cameras [17/05 15:14:11]
Loading Test Cameras [17/05 15:14:13]
Number of points at initialisation :  11043 [17/05 15:14:13]
Training progress:  23% 7000/30000 [01:26<05:13, 73.35it/s, Loss=0.0107366]
[ITER 7000] Evaluating train: L1 0.0053535101003944876 PSNR 31.519995117187502 [17/05 15:15:40]

[ITER 7000] Saving Gaussians [17/05 15:15:40]
Training progress: 100% 30000/30000 [07:52<00:00, 63.48it/s, Loss=0.0060012]

[ITER 30000] Evaluating train: L1 0.003392478544265032 PSNR 33.773607635498045 [17/05 15:22:06]

[ITER 30000] Saving Gaussians [17/05 15:22:06]

Training c

In [27]:
%cd "/content/drive/MyDrive/Assignment 4 - 3DGS"

!python3.10 export_ply.py \
    --checkpoint data/chair/checkpoints/checkpoint_000180.pt \
    --output data/chair/chair_model.ply

/content/drive/MyDrive/Assignment 4 - 3DGS
正在加载权重文件: data/chair/checkpoints/checkpoint_000180.pt ...
正在构建 PLY 结构并写入文件...
转换成功！标准的 3DGS 文件已保存至: data/chair/chair_model.ply


In [28]:
%cd "/content/drive/MyDrive/Assignment 4 - 3DGS"

!python3.10 export_ply.py \
    --checkpoint data/lego/checkpoints/checkpoint_000080.pt \
    --output data/lego/lego_model.ply

/content/drive/MyDrive/Assignment 4 - 3DGS
正在加载权重文件: data/lego/checkpoints/checkpoint_000080.pt ...
正在构建 PLY 结构并写入文件...
转换成功！标准的 3DGS 文件已保存至: data/lego/lego_model.ply
